Run the below function for the parameter values:

D = 1/7 to D = 1/9 for sf = 2.5
sf = 1.5 to sf = 2.5 for D = 1/7.3

I observe the following:
(1) Bifurcation somewhere between D=1/7.2 and D=1/7.3 for sf = 2.5
(2) Bifurcation somewhere between sf = 1.7 and sf = 2.0 for D = 1/7.3
(3) For sf=2.5, when D<1/8.5, the 2nd variable washes out (near 0), but somewhere between D=1/8.5 and D=1/9, this variable 'comes back to life' and becomes non-zero again

In [1]:
import numpy as np

def ode_fun(t, var, par):
    x, y, z, u, v, g = var
    alpha, uf, omega, sigma, rho, eta, phi1, phi2, uc1_prime, uc2_prime, uc3_prime = par

    u1_prime = u / ((1+u) * (1+g))
    u2_prime = phi1 * v / (1+v)
    u3_prime = phi2 * v / (sigma + v)

    output = np.zeros(6)
    
    output[0] = -alpha * x + u1_prime * x - uc1_prime * x
    output[1] = -alpha * y + u2_prime * y - uc2_prime * y
    output[2] = -alpha * z + u3_prime * z - uc3_prime * z
    output[3] = alpha * (uf - u) - u1_prime * x
    output[4] = -alpha * v + omega * u1_prime * x - u2_prime * y - sigma * u3_prime * z
    output[5] = -alpha * g + rho * u2_prime * y + eta * u3_prime * z

    return output

def torch_ode_fun(t, var, par):
    x, y, z, u, v, g = var
    alpha, uf, omega, sigma, rho, eta, phi1, phi2, uc1_prime, uc2_prime, uc3_prime = par

    u1_prime = u / ((1+u) * (1+g))
    u2_prime = phi1 * v / (1+v)
    u3_prime = phi2 * v / (sigma + v)

    output = torch.zeros(6)
    
    output[0] = -alpha * x + u1_prime * x - uc1_prime * x
    output[1] = -alpha * y + u2_prime * y - uc2_prime * y
    output[2] = -alpha * z + u3_prime * z - uc3_prime * z
    output[3] = alpha * (uf - u) - u1_prime * x
    output[4] = -alpha * v + omega * u1_prime * x - u2_prime * y - sigma * u3_prime * z
    output[5] = -alpha * g + rho * u2_prime * y + eta * u3_prime * z

    return output



def par_fun(u_m1=0.68, u_m2=0.20, u_m3=0.25, u_c1=0.25, u_c2=0.08, u_c3=0.11,
            K_1=0.01, K_2=0.001, K_3=0.01, K_i=0.01, Y_1=1/2, Y_2=1/1.37, Y_3=1/1.2,
            beta=1.94, gamma=1.8, delta=1.55, D=1/14, sf=1.0):
    
    alpha       =   D / u_m1
    uf          =   sf / K_1
    omega       =   beta * Y_1 * K_1 / K_2
    sigma       =   K_3 / K_2
    rho         =   gamma * Y_2 * K_2 / K_i
    eta         =   delta * Y_3 * K_3 / K_i
    phi1        =   u_m2 / u_m1
    phi2        =   u_m3 / u_m1
    uc1_prime   =   u_c1 / u_m1
    uc2_prime   =   u_c2 / u_m1
    uc3_prime   =   u_c3 / u_m1

    return alpha, uf, omega, sigma, rho, eta, phi1, phi2, uc1_prime, uc2_prime, uc3_prime 
    

In [2]:
# abc = []
# for i in range(6):
#     abc.append([])
# print(abc)
# abc[0].append(6)
# abc[3].append(3)
# print(abc)

In [44]:
%cd /home/smalani/PartialObservations/

from scipy.integrate import solve_ivp
from PartialObservations.BandFModel import num_integrator
from tqdm.auto import tqdm
from scipy.optimize import root
import torch
import matplotlib.pyplot as plt
from torchdiffeq import odeint

mypar = par_fun(D=1/7.3, sf=100)
t = 0
# myvar0 = np.array([2.45524443e+01, 9.40042585e-11, 1.94133806e-01, 1.96344728e+02,
#  5.16882041e+02, 4.61654478e-01])
# myvar0 = [7.54651217e+001, 5.68061229e-110, 3.16832990e-001, 3.07716331e+001,
#  2.12083688e+003, 7.33444520e-001]

# myvar0 = [7.83577452e+01, 1.76910142e-01, 1.21884731e-02, 1.24296145e+00,
#  2.41269019e+03, 3.27558469e-02]

# myvar0 = np.concatenate((np.random.uniform(0,100,1),
#                          np.random.uniform(0,1.5,1),
#                          np.random.uniform(0,0,1),
#                          np.random.uniform(0,250,1),
#                          np.random.uniform(0,2200,1),
#                          np.random.uniform(0,4.5,1)))

f = np.load('minmax/initialcond.npz')
myvar0 = f['arr_0']
index = np.random.choice(myvar0.shape[1])
my_range = (np.max(myvar0, axis=1) - np.min(myvar0, axis=1))[0,:]
mean = ((np.max(myvar0, axis=1) - np.min(myvar0, axis=1)) / 2 + np.min(myvar0, axis=1))[0,:]
init = myvar0[0, index, :]
loc = (mean - init) / 2
# loc[1:] = 0

loc[1] = 0.1
my_range[1] = 0.5

init = np.abs(init + np.random.normal(loc=loc, scale=my_range/10))

# zero_sol = fsolve(lambda y, par: ode_fun(0,y,par), myvar0, args=(mypar,))
# print(zero_sol)

dt = 1
tspan = [0,1000]
tskip = 0
tarr = np.arange(tskip,tspan[1]+tskip,step=0.1)

sol = solve_ivp(ode_fun, y0=init, 
                t_span=[tspan[0],tspan[1]+tskip], t_eval=tarr, args=(mypar,), 
                rtol=1e-10, atol=1e-10)
sol_RK2_t, sol_RK2_x = num_integrator.RK2_int(ode_fun, tspan, sol.y[:,0], mypar, dt)
sol_RK4_t, sol_RK4_x = num_integrator.RK4_int(ode_fun, tspan, sol.y[:,0], mypar, dt)
# sol_DOPRI_t, sol_DOPRI_x = num_integrator.DOPRI_int(ode_fun, tspan, sol.y[:,0], mypar, dt)
# sol_torchodeint_t, sol_torchodeint_x = num_integrator.torch_odeint(torch_ode_fun, tspan, sol.y[:,0], mypar, dt)

labels = ['x', 'y', 'z', 'u', 'v', 'g']

print(sol.y[:,-1])

plt.figure(figsize=(20,10))

for i in range(6):
    # print('bloop')
    ax = plt.subplot(int(str(61) + str(i+1)))
    ax.plot(sol.t - sol.t[0], sol.y[i,:], label='solve_ivp',color='b')
    ax2 = ax.twinx()
    ax2.plot(sol_RK4_t, sol_RK4_x[i,:], label='RK4',color='r')
    # ax2.plot(sol_RK2_t, sol_RK2_x[i,:], label='RK2',color='g')
    # ax2.plot(sol_DOPRI_t, sol_DOPRI_x[i,:], label='DOPRI')
    # plt.plot(sol_torchodeint_t, sol_torchodeint_x[i,:], label='torch')
    ax.set_ylabel(labels[i] + ' true', color='b')
    ax2.set_ylabel(labels[i] + ' RK4', color='r')
    plt.legend()
plt.xlabel(r'theta')
plt.show()



/home/smalani/PartialObservations


In [4]:
# from scipy.integrate import solve_ivp
# from tqdm.auto import tqdm
# from scipy.optimize import root
# import torch

# theta_arr = np.linspace(9,10,200)

# output = np.array([5.07443675e+01, 1.72061852e-91, 2.00195776e-01, 1.39158972e+02,
#  1.07152606e+03, 4.69122154e-01])

# stable_ss = []
# stable_theta = []

# unstable_ss = []
# unstable_theta = []

# max_vals_theta = []
# min_vals_theta = []

# for i, theta in enumerate(tqdm(theta_arr)):
#     mypar = par_fun(D=1/theta, sf=2.5)

#     init = output.copy()
#     init[init < 1e-5] = 1

#     sol = root(lambda y, par: ode_fun(0,y,par), output, args=(mypar,),
#                     method='hybr',)
#                     #  options={'factor': 0.1})
#     output = sol.x

#     J = torch.autograd.functional.jacobian(lambda y: torch_ode_fun(0, y, mypar),
#              torch.from_numpy(output), strict=True)
#     w, v = np.linalg.eig(J)

#     if ~(np.any(w.real>0)):
#         stable_ss.append(output)
#         stable_theta.append(theta)
#     else:
#         unstable_ss.append(output)
#         unstable_theta.append(theta)

#     # if i%10 == 9:
#     #     print(i)


In [5]:
# import matplotlib.pyplot as plt

# unstable_ss = np.array(unstable_ss)
# stable_ss = np.array(stable_ss)
# unstable_theta = np.array(unstable_theta)
# stable_theta = np.array(stable_theta)

# print(stable_theta.shape)
# print(unstable_theta.shape)

# plt.figure(figsize=(20,10))

# labels = ['x', 'y', 'z', 'u', 'v', 'g']
# for i in range(6):
#     plt.subplot(int(str(61) + str(i+1)))
#     plt.plot(stable_theta, stable_ss[:,i],'k-')
#     plt.plot(unstable_theta, unstable_ss[:,i],'k--')
#     plt.ylabel(labels[i])
# plt.xlabel(r'theta')

In [6]:
# from scipy.integrate import solve_ivp
# from tqdm.auto import tqdm

# theta_arr = np.linspace(7,10,200)

# output = np.array([5.07443675e+01, 1.72061852e-91, 2.00195776e-01, 1.39158972e+02,
#  1.07152606e+03, 4.69122154e-01])

# stable_ss = []
# stable_theta = []

# unstable_ss = []
# unstable_theta = []

# max_vals_theta = []
# min_vals_theta = []

# for i, theta in enumerate(tqdm(theta_arr)):

#     # print(i)
#     t_span = [0, 10000]
#     t_eval = np.linspace(t_span[1]*0.9, t_span[1], 10000)
    
#     mypar = par_fun(D=1/theta, sf=2.5)
#     sol = solve_ivp(ode_fun,t_span, output, t_eval=t_eval, args=(mypar,), rtol=1e-5, atol=1e-8)

#     max_vals_theta.append(np.max(sol.y,axis=1))
#     min_vals_theta.append(np.min(sol.y,axis=1))
#     output = sol.y[:,-1] + 0.001

#     # if i%10 == 9:
#     #     print(i)


In [7]:
# import matplotlib.pyplot as plt

# max_vals_theta = np.array(max_vals_theta)
# min_vals_theta = np.array(min_vals_theta)

# print(max_vals_theta.shape)

# plt.figure(figsize=(20,10))

# labels = ['x', 'y', 'z', 'u', 'v', 'g']
# for i in range(6):
#     plt.subplot(int(str(61) + str(i+1)))
#     plt.plot(theta_arr, max_vals_theta[:,i],'k-')
#     plt.plot(theta_arr, min_vals_theta[:,i],'k-')
#     plt.ylabel(labels[i])
# plt.xlabel(r'theta')

In [8]:
# from scipy.integrate import solve_ivp
# from tqdm.auto import tqdm
# from scipy.optimize import root

# sf_arr = np.concatenate((np.linspace(1.5,1.79,20),
#                          np.linspace(1.79,1.81,50),
#                          np.linspace(1.81,2,5,50)))

# output = np.array([5.07443675e+01, 1.72061852e-91, 2.00195776e-01, 1.39158972e+02,
#  1.07152606e+03, 4.69122154e-01])

# stable_ss = []
# stable_theta = []

# unstable_ss = []
# unstable_theta = []

# max_vals_sf = []
# min_vals_sf = []

# for i, sf in enumerate(tqdm(sf_arr)):
#     t_span = [0, 10000]
#     t_eval = np.linspace(t_span[1]*0.9, t_span[1], 10000)
    
#     mypar = par_fun(D=1/7.3, sf=sf)
#     sol = solve_ivp(ode_fun,t_span, output, t_eval=t_eval, args=(mypar,), rtol=1e-5, atol=1e-8)

#     max_vals_sf.append(np.max(sol.y,axis=1))
#     min_vals_sf.append(np.min(sol.y,axis=1))
#     output = sol.y[:,-1] + 0.001



#     # if i%10 == 9:
#     #     print(i)


In [9]:
# import matplotlib.pyplot as plt

# max_vals_sf = np.array(max_vals_sf)
# min_vals_sf = np.array(min_vals_sf)

# plt.figure(figsize=(20,10))

# labels = ['x', 'y', 'z', 'u', 'v', 'g']
# for i in range(6):
#     plt.subplot(int(str(61) + str(i+1)))
#     plt.plot(sf_arr, max_vals_sf[:,i],'k-')
#     plt.plot(sf_arr, min_vals_sf[:,i],'k-')
#     plt.ylabel(labels[i])
# plt.xlabel(r'theta')

In [10]:
# np.savez('PartialObservations/BandFModel/Bifurc_minmaxes',
#          max_vals_theta=max_vals_theta,
#          min_vals_theta=min_vals_theta,
#          max_vals_sf=max_vals_sf,
#          min_vals_sf=min_vals_sf)